# Télécharger des vidéos de mouvement humain depuis YouTube
## Tutoriel 1 / 2

Ce notebook montre comment rechercher et télécharger automatiquement des vidéos
à partir de mots-clés, pour constituer un dataset de mouvements humains.

**Pipeline :**
```
Mots-clés  →  Recherche YouTube  →  Filtrage  →  Téléchargement  →  Dataset local
```

> **Suite :** Le tutoriel 2 (`02_mediapipe_pose.ipynb`) utilise ces vidéos
> pour extraire les poses 3D avec MediaPipe.

In [ ]:
# Installation
!pip install yt-dlp --quiet
!pip install yt-dlp[default] --quiet   # backends audio/vidéo

import yt_dlp
import os, json, re
from pathlib import Path
from IPython.display import HTML, display

print(f"yt-dlp version : {yt_dlp.version.__version__}")

## 1. Définir les mots-clés de recherche

Modifiez `SEARCH_QUERIES` selon votre sport ou mouvement cible.
Chaque entrée peut contenir plusieurs mots-clés alternatifs.

In [ ]:
# ── À MODIFIER ──────────────────────────────────────────────────────────────
SEARCH_QUERIES = [
    "ice skating technique slow motion",
    "speed skating side view",
    "figure skating jump slow motion",
    "inline skating tutorial",
]

MAX_RESULTS_PER_QUERY = 5    # vidéos par mot-clé
MAX_DURATION_SEC      = 300  # ignorer les vidéos > 5 min
MIN_DURATION_SEC      = 10   # ignorer les vidéos < 10 sec
OUTPUT_DIR            = Path("videos")   # dossier de sortie
# ────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Dossier de sortie : {OUTPUT_DIR.resolve()}")
print(f"{len(SEARCH_QUERIES)} requête(s) × {MAX_RESULTS_PER_QUERY} vidéos = "
      f"jusqu'à {len(SEARCH_QUERIES)*MAX_RESULTS_PER_QUERY} vidéos")

## 2. Recherche — récupérer les métadonnées sans télécharger

On commence par **lister** les vidéos candidates avec leurs métadonnées
(durée, titre, vues) pour pouvoir filtrer avant de télécharger.

In [ ]:
def search_videos(query, max_results=5):
    """
    Cherche des vidéos YouTube et retourne leurs métadonnées.
    N'effectue aucun téléchargement.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "extract_flat": True,       # métadonnées seulement, pas de download
        "playlistend": max_results,
    }

    url = f"ytsearch{max_results}:{query}"

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    results = []
    for entry in info.get("entries", []):
        if entry is None:
            continue
        results.append({
            "id":       entry.get("id", ""),
            "title":    entry.get("title", "?"),
            "duration": entry.get("duration") or 0,
            "view_count": entry.get("view_count") or 0,
            "url":      f"https://www.youtube.com/watch?v={entry.get('id', '')}",
        })
    return results


# Lancer la recherche pour toutes les requêtes
all_candidates = {}   # query → liste de vidéos

for query in SEARCH_QUERIES:
    print(f"\n🔍 Recherche : '{query}'")
    results = search_videos(query, MAX_RESULTS_PER_QUERY)
    all_candidates[query] = results
    for i, v in enumerate(results):
        dur = f"{v['duration']//60}:{v['duration']%60:02d}"
        print(f"  {i+1}. [{dur}] {v['title'][:70]}")

total = sum(len(v) for v in all_candidates.values())
print(f"\n→ {total} vidéos candidates trouvées")

## 3. Filtrage automatique

On conserve uniquement les vidéos dans la plage de durée souhaitée
et on déduplique (une même vidéo peut apparaître dans plusieurs requêtes).

In [ ]:
seen_ids  = set()
to_download = []

for query, videos in all_candidates.items():
    for v in videos:
        dur = v["duration"]
        vid = v["id"]

        if vid in seen_ids:
            continue                          # doublon
        if dur < MIN_DURATION_SEC:
            continue                          # trop courte
        if dur > MAX_DURATION_SEC:
            continue                          # trop longue

        seen_ids.add(vid)
        to_download.append({**v, "query": query})

print(f"{len(to_download)} vidéos retenues après filtrage :\n")
for i, v in enumerate(to_download):
    dur = f"{v['duration']//60}:{v['duration']%60:02d}"
    print(f"  {i+1:2d}. [{dur}]  {v['title'][:65]}")
    print(f"       requête : {v['query']}")

## 4. Sélection manuelle (optionnel)

Décommentez la cellule suivante pour choisir manuellement quelles vidéos garder.

In [ ]:
# ── Sélection manuelle — décommentez pour activer ──────────────────────────
# KEEP_INDICES = [1, 2, 4, 7]   # numéros de la liste ci-dessus (1-indexés)
# to_download  = [to_download[i-1] for i in KEEP_INDICES]
# print(f"{len(to_download)} vidéos sélectionnées manuellement")
# ────────────────────────────────────────────────────────────────────────────
print("Sélection automatique conservée.")

## 5. Téléchargement

Format cible : **MP4 480p** (bon compromis qualité/taille pour la pose estimation).
Les vidéos déjà téléchargées sont automatiquement ignorées.

In [ ]:
def sanitize_filename(title, max_len=60):
    """Nettoie un titre YouTube pour en faire un nom de fichier valide."""
    name = re.sub(r'[\\/*?":.<>|]', "", title)   # caractères interdits
    name = re.sub(r'\s+', ' ', name).strip()       # espaces multiples
    return name[:max_len]


def download_video(video_info, output_dir, max_height=480):
    """
    Télécharge une vidéo en MP4 avec résolution max max_height.
    Retourne le chemin du fichier téléchargé, ou None si échec.
    """
    title    = sanitize_filename(video_info["title"])
    out_path = output_dir / f"{title}.mp4"

    if out_path.exists():
        print(f"  [SKIP] déjà téléchargée : {out_path.name}")
        return str(out_path)

    ydl_opts = {
        "quiet":   True,
        "no_warnings": True,
        # Meilleur MP4 ≤ max_height, ou le meilleur disponible
        "format":  f"bestvideo[height<={max_height}][ext=mp4]+bestaudio[ext=m4a]/"
                   f"best[height<={max_height}][ext=mp4]/best[ext=mp4]/best",
        "merge_output_format": "mp4",
        "outtmpl": str(output_dir / f"{title}.%(ext)s"),
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_info["url"]])
        # Renomme si l'extension finale n'est pas .mp4
        candidates = list(output_dir.glob(f"{title}.*"))
        if candidates and candidates[0].suffix != ".mp4":
            candidates[0].rename(out_path)
        return str(out_path)
    except Exception as e:
        print(f"  [ERREUR] {video_info['title'][:50]} : {e}")
        return None


# Téléchargement de toutes les vidéos retenues
downloaded = []
for i, video in enumerate(to_download):
    print(f"\n[{i+1}/{len(to_download)}] {video['title'][:65]}")
    path = download_video(video, OUTPUT_DIR)
    if path:
        size_mb = os.path.getsize(path) / 1e6
        print(f"  → {path}  ({size_mb:.1f} MB)")
        downloaded.append({**video, "path": path, "size_mb": size_mb})

print(f"\n✓ {len(downloaded)}/{len(to_download)} vidéos téléchargées")

## 6. Résumé du dataset

In [ ]:
total_dur_s  = sum(v["duration"] for v in downloaded)
total_size   = sum(v["size_mb"]  for v in downloaded)

print(f"Dataset : {len(downloaded)} vidéos")
print(f"  Durée totale : {total_dur_s//60} min {total_dur_s%60} s")
print(f"  Taille totale : {total_size:.0f} MB")
print(f"  Dossier : {OUTPUT_DIR.resolve()}")
print()

# Sauvegarde des métadonnées
meta_path = OUTPUT_DIR / "metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(downloaded, f, ensure_ascii=False, indent=2)
print(f"Métadonnées sauvées : {meta_path}")

print("\nListe des vidéos :")
for v in downloaded:
    dur = f"{v['duration']//60}:{v['duration']%60:02d}"
    print(f"  [{dur}]  {v['title'][:65]}")

## Étape suivante

Ouvrez **`02_mediapipe_pose.ipynb`** pour extraire les poses 3D
depuis les vidéos que vous venez de télécharger.

```python
# Dans le tuto 2, pointez vers votre dossier :
VIDEO_DIR = Path("videos")
```